In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")


In [ ]:
subset_size = 64
dataset = load_dataset("glue", "mrpc", split=f"validation[:{subset_size}]")
print(f"Validation subset examples: {len(dataset)}")

preview_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print(preview_df.head(5).to_string(index=False))


In [ ]:
batch_size = 16
predictions = []
true_labels = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1)

    predictions.extend(preds.cpu().tolist())
    true_labels.extend(batch["label"])

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": f"validation[:{subset_size}]",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device)
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["is_correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

correct_df = examples_df[examples_df["is_correct"]].copy()
incorrect_df = examples_df[~examples_df["is_correct"]].copy()

print(f"\nCorrect predictions: {len(correct_df)}")
if len(correct_df) > 0:
    print(correct_df.head(10).to_string(index=False))

print(f"\nIncorrect predictions: {len(incorrect_df)}")
if len(incorrect_df) > 0:
    print(incorrect_df.head(10).to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
